# Live Testing Notebook: RSFM Integration on Kambera Study Area

This notebook runs the `ML_LC_Classifier.rsfm` workflow using the real **Kambera (East Nusa Tenggara)** vector & schema dataset located at `input_data/Kambera/`:
- **AOI:** `AOI_Kambera_East_Nusa_Tenggara.shp` (7,913.34 ha)
- **Training samples:** `Training_data_kambera.shp` (460 points, label column `ID_Kelas`)
- **Independent validation samples:** `validation_data_kambera.shp` (459 points, label column `ID_Kelas`)
- **Schema:** `MODUL2_Skema_Klasifikasi_Kambera.csv` (9 land-cover classes + color palette)

### Execution Modes
1. **Imagery Source:**
   - If you place real GeoTIFFs inside `input_data/Kambera/` (or set `CUSTOM_RASTER_T1`, `CUSTOM_DEM`, etc. below), the notebook runs on your real satellite rasters.
   - Set `AUTO_DOWNLOAD_STAC = True` (requires `pip install -e ".[experimental]"`) to fetch real Sentinel-2 L2A & Copernicus DEM over the Kambera AOI from Microsoft Planetary Computer.
   - If no GeoTIFF is present locally yet, the notebook builds a spatially co-registered synthetic raster stack matching the **exact Kambera UTM 51S bounds** and **real 460 training / 459 validation point locations** so every cell runs end-to-end immediately.
2. **Foundation Model Backbone:**
   - `USE_OPENGEOAI = False`: runs locally on CPU with `SimpleCPUBackbone`.
   - `USE_OPENGEOAI = True`: loads real foundation models from OpenGeoAI (`pip install -e ".[rsfm]"` on Colab / GPU).

## 1. Environment Setup, Kambera Paths & Mode Toggles

In [ ]:
from pathlib import Path
import sys

project_root = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir()
)
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# --- CONFIGURATION TOGGLES ---
# 1. Set USE_OPENGEOAI = True on Colab / GPU after running: pip install -e ".[rsfm]"
USE_OPENGEOAI = False
OPENGEOAI_MODEL_NAME = "prithvi_eo_v2_300"

# 2. Set AUTO_DOWNLOAD_STAC = True (requires .[experimental]) to fetch Sentinel-2 & DEM for Kambera
AUTO_DOWNLOAD_STAC = False

# 3. Optional explicit raster paths (leave None to auto-detect *.tif in input_data/Kambera or build aligned test stack)
CUSTOM_RASTER_T1 = None
CUSTOM_RASTER_T2 = None
CUSTOM_DEM = None
CUSTOM_SNIC_SEG = None

# --- KAMBERA REAL DATA PATHS ---
KAMBERA_DIR = project_root / "input_data" / "Kambera"
AOI_PATH = KAMBERA_DIR / "AOI_Kambera_East_Nusa_Tenggara.shp"
TRAIN_PATH = KAMBERA_DIR / "Training_data_kambera.shp"
VAL_PATH = KAMBERA_DIR / "validation_data_kambera.shp"
SCHEMA_PATH = KAMBERA_DIR / "MODUL2_Skema_Klasifikasi_Kambera.csv"
CLASS_FIELD = "ID_Kelas"

WORK_DIR = project_root / "output" / "rsfm_kambera_test"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Kambera directory: {KAMBERA_DIR}")
print(f"Work/output directory: {WORK_DIR}")

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
from scipy.ndimage import distance_transform_edt

from ML_LC_Classifier import (
    SimpleCPUBackbone,
    apply_transition_consistency,
    assess_map_accuracy,
    assign_segment_labels,
    bayesian_smooth,
    classify_from_probs,
    compute_embedding_stability,
    extract_rsfm_embeddings,
    filter_disturbed_samples,
    finetune_rsfm,
    infer_rsfm_probabilities,
    list_opengeoai_models,
    load_and_extract,
    load_opengeoai_backbone,
    load_rsfm_checkpoint,
    pool_rsfm_by_segments,
    prepare_rsfm_chips,
    process_tile,
    project_segment_probabilities,
    tune_model,
)

schema_df = pd.read_csv(SCHEMA_PATH)
KAMBERA_CLASS_NAMES = dict(zip(schema_df["ID"].astype(int), schema_df["Land Cover Class"]))
KAMBERA_COLORS = dict(zip(schema_df["ID"].astype(int), schema_df["Color Palette"]))
display_schema = schema_df.copy()
print(display_schema.to_string(index=False))

## 2. Initialize Backbone (OpenGeoAI vs. Local CPU Test Backbone)

In [ ]:
if USE_OPENGEOAI:
    available = list_opengeoai_models()
    print(f"OpenGeoAI models available: {len(available)}")
    backbone = load_opengeoai_backbone(OPENGEOAI_MODEL_NAME)
    CHIP_SIZE = 224
    STRIDE = 112
else:
    backbone = SimpleCPUBackbone(embed_dim=16, seed=42)
    CHIP_SIZE = 16
    STRIDE = 8

print(f"Active backbone: {type(backbone).__name__} (embed_dim={backbone.embed_dim})")

## 3. Load Real Kambera Vectors & Resolve/Prepare Co-Registered Kambera Rasters

Reads `AOI_Kambera_East_Nusa_Tenggara.shp`, `Training_data_kambera.shp`, and `validation_data_kambera.shp`. If GeoTIFFs are not yet placed in `input_data/Kambera/`, optionally downloads Sentinel-2 + DEM via Planetary Computer (`AUTO_DOWNLOAD_STAC = True`) or constructs an aligned UTM Zone 51S test grid over the real Kambera AOI and sample locations.

In [ ]:
aoi_gdf = gpd.read_file(AOI_PATH)
train_gdf = gpd.read_file(TRAIN_PATH)
val_gdf = gpd.read_file(VAL_PATH)

print(f"AOI: {aoi_gdf['nmkec'].iloc[0]} | Area: {aoi_gdf['ha'].iloc[0]:.2f} ha | CRS: {aoi_gdf.crs}")
print(f"Training points: {len(train_gdf)} | Validation points: {len(val_gdf)}")
print("Training class distribution:\n", train_gdf[CLASS_FIELD].value_counts().sort_index())

existing_tifs = sorted(KAMBERA_DIR.glob("*.tif"))
if existing_tifs:
    print("Found GeoTIFFs in input_data/Kambera:", [p.name for p in existing_tifs])

if CUSTOM_RASTER_T1 is not None:
    s2_t1_path = Path(CUSTOM_RASTER_T1)
    s2_t2_path = Path(CUSTOM_RASTER_T2) if CUSTOM_RASTER_T2 else s2_t1_path
    dem_path = Path(CUSTOM_DEM) if CUSTOM_DEM else None
    seg_path = Path(CUSTOM_SNIC_SEG) if CUSTOM_SNIC_SEG else None
    USING_REAL_IMAGERY = True
elif AUTO_DOWNLOAD_STAC:
    from ML_LC_Classifier.experimental.planetary_computer import (
        create_feature_stack,
        download_imagery,
        search_imagery,
    )
    s2_scenes_t1 = search_imagery(
        aoi=AOI_PATH,
        collection="sentinel-2-l2a",
        start_date="2023-05-01",
        end_date="2023-10-31",
        max_cloud_cover=20,
    )
    s2_assets_t1 = download_imagery(s2_scenes_t1[:4], assets=["B02", "B03", "B04", "B08"], output_dir=WORK_DIR / "stac_s2_t1")
    s2_t1_path = create_feature_stack(s2_assets_t1, WORK_DIR / "kambera_s2_2023.tif", composite="median", target_crs="EPSG:32751", resolution=20.0).stack_path
    s2_scenes_t2 = search_imagery(
        aoi=AOI_PATH,
        collection="sentinel-2-l2a",
        start_date="2024-05-01",
        end_date="2024-10-31",
        max_cloud_cover=20,
    )
    s2_assets_t2 = download_imagery(s2_scenes_t2[:4], assets=["B02", "B03", "B04", "B08"], output_dir=WORK_DIR / "stac_s2_t2")
    s2_t2_path = create_feature_stack(s2_assets_t2, WORK_DIR / "kambera_s2_2024.tif", composite="median", target_crs="EPSG:32751", resolution=20.0).stack_path
    dem_scenes = search_imagery(aoi=AOI_PATH, collection="cop-dem-glo-30")
    dem_assets = download_imagery(dem_scenes[:2], assets=["data"], output_dir=WORK_DIR / "stac_dem")
    dem_path = create_feature_stack(dem_assets, WORK_DIR / "kambera_dem.tif", asset_aliases={"data": "dem"}, terrain_from=["dem"]).stack_path
    seg_path = None
    USING_REAL_IMAGERY = True
else:
    # Build a 30m UTM Zone 51S raster stack covering the exact Kambera AOI bounds
    # and driven by the real Training_data_kambera.shp point distribution.
    TARGET_CRS = "EPSG:32751"
    PIXEL_SIZE = 30.0
    aoi_utm = aoi_gdf.to_crs(TARGET_CRS)
    train_utm = train_gdf.to_crs(TARGET_CRS)
    minx, miny, maxx, maxy = aoi_utm.total_bounds
    pad = 480.0
    minx, miny, maxx, maxy = minx - pad, miny - pad, maxx + pad, maxy + pad
    W = int(np.ceil((maxx - minx) / PIXEL_SIZE))
    H = int(np.ceil((maxy - miny) / PIXEL_SIZE))
    utm_transform = from_origin(minx, maxy, PIXEL_SIZE, PIXEL_SIZE)

    # Nearest-training-point Voronoi surface across Kambera AOI so real points map to realistic spectra
    pts_mask = np.zeros((H, W), dtype=np.int32)
    for _, row in train_utm.iterrows():
        r, c = rasterio.transform.rowcol(utm_transform, row.geometry.x, row.geometry.y)
        if 0 <= r < H and 0 <= c < W:
            pts_mask[r, c] = int(row[CLASS_FIELD])
    _, nearest_idx = distance_transform_edt(pts_mask == 0, return_indices=True)
    voronoi_cls = pts_mask[nearest_idx[0], nearest_idx[1]]

    # Class-specific Sentinel-2 surface reflectance prototypes (B02, B03, B04, B08) + elevation
    class_spectra = {
        1: [0.03, 0.05, 0.03, 0.44],  # Hutan Lahan Kering
        2: [0.03, 0.06, 0.03, 0.36],  # Hutan Mangrove
        3: [0.04, 0.07, 0.04, 0.38],  # Kebun Campur
        4: [0.07, 0.10, 0.09, 0.27],  # Padang Rumput
        5: [0.08, 0.11, 0.11, 0.24],  # Pertanian Lahan Kering
        6: [0.05, 0.09, 0.06, 0.31],  # Sawah
        7: [0.16, 0.18, 0.21, 0.17],  # Lahan Terbangun
        8: [0.21, 0.24, 0.27, 0.15],  # Lahan Terbuka
        9: [0.06, 0.07, 0.04, 0.03],  # Badan Air
    }
    class_elev = {1: 420.0, 2: 2.0, 3: 180.0, 4: 140.0, 5: 110.0, 6: 35.0, 7: 25.0, 8: 90.0, 9: 5.0}

    rng = np.random.default_rng(42)
    s2_t1 = rng.normal(0.0, 0.004, size=(4, H, W)).astype(np.float32)
    elev_arr = rng.normal(0.0, 2.0, size=(1, H, W)).astype(np.float32)
    for cid, spec in class_spectra.items():
        m = voronoi_cls == cid
        for b_i, val in enumerate(spec):
            s2_t1[b_i, m] += val
        elev_arr[0, m] += class_elev[cid]

    # Epoch t2: simulate localized conversion in south-eastern quadrant
    s2_t2 = s2_t1.copy()
    change_mask = (np.arange(H)[:, None] > int(0.65 * H)) & (np.arange(W)[None, :] > int(0.60 * W))
    for b_i, val in enumerate(class_spectra[8]):
        s2_t2[b_i, change_mask] = val + rng.normal(0.0, 0.004, size=int(change_mask.sum())).astype(np.float32)

    s2_t1_path = WORK_DIR / "kambera_s2_t1.tif"
    s2_t2_path = WORK_DIR / "kambera_s2_t2.tif"
    for p, arr in [(s2_t1_path, s2_t1), (s2_t2_path, s2_t2)]:
        with rasterio.open(
            p, "w", driver="GTiff", height=H, width=W, count=4,
            dtype="float32", crs=TARGET_CRS, transform=utm_transform, nodata=np.nan
        ) as dst:
            dst.write(arr)
            dst.descriptions = ("B02", "B03", "B04", "B08")

    dem_path = WORK_DIR / "kambera_dem.tif"
    with rasterio.open(
        dem_path, "w", driver="GTiff", height=H, width=W, count=1,
        dtype="float32", crs=TARGET_CRS, transform=utm_transform, nodata=np.nan
    ) as dst:
        dst.write(elev_arr)
        dst.descriptions = ("NASADEM_HGT",)

    # Regular 12x12 pixel (~360m) superpixel grid with wrapped negative int32 GEE IDs on the bottom half
    seg_path = WORK_DIR / "kambera_snic_segments.tif"
    tile_r = np.arange(H)[:, None] // 12
    tile_c = np.arange(W)[None, :] // 12
    seg_grid = (tile_r * 100 + tile_c + 1).astype(np.int32)
    seg_grid[H // 2 :, :] = -2000000000 - seg_grid[H // 2 :, :]
    with rasterio.open(
        seg_path, "w", driver="GTiff", height=H, width=W, count=1,
        dtype="int32", crs=TARGET_CRS, transform=utm_transform, nodata=0
    ) as dst:
        dst.write(seg_grid, 1)

    USING_REAL_IMAGERY = False
    print(f"[FALLBACK MODE] Created aligned {H}x{W} (30m UTM 51S) Kambera test stack.")

## 4. Workflow 1: Windowed Embedding Stack -> Train on `Training_data_kambera.shp` -> Evaluate on `validation_data_kambera.shp`

In [ ]:
emb_result = extract_rsfm_embeddings(
    s2_t1_path,
    WORK_DIR / "kambera_embeddings_t1.tif",
    backbone=backbone,
    chip_size=CHIP_SIZE,
    stride=STRIDE,
    wavelengths_um=[0.49, 0.56, 0.66, 0.84],
)
print(f"Embedding stack: {emb_result.stack_path.name} ({len(emb_result.feature_names)} bands)")

ds_tr, X_train, y_train = load_and_extract(emb_result.stack_path, TRAIN_PATH, class_field=CLASS_FIELD)
ds_tr.close()
ds_val, X_val, y_val = load_and_extract(emb_result.stack_path, VAL_PATH, class_field=CLASS_FIELD)
ds_val.close()

rf_tuned = tune_model(
    classifier="RandomForest",
    parameter_space={"n_estimators": [100]},
    X_train=X_train,
    y_train=y_train,
    cv=3,
    random_state=42,
)
val_acc = float((rf_tuned.model.predict(X_val) == y_val).mean())
print(f"Independent validation_data_kambera.shp accuracy (RSFM Embeddings + RF, n={len(y_val)}): {val_acc:.2%}")

## 5. Workflow 2: Sparse-Point LoRA Fine-Tuning on `Training_data_kambera.shp`

In [ ]:
chip_dataset = prepare_rsfm_chips(
    s2_t1_path,
    TRAIN_PATH,
    class_field=CLASS_FIELD,
    chip_size=CHIP_SIZE,
    wavelengths_um=[0.49, 0.56, 0.66, 0.84],
)
print(f"Extracted chips: {chip_dataset.chips.shape} | Classes: {chip_dataset.class_values}")

ft_result = finetune_rsfm(
    chip_dataset,
    backbone=backbone,
    lora_rank=4,
    lora_alpha=8.0,
    epochs=40,
    lr=0.15,
)
print(f"LoRA masked loss: {ft_result.loss_history[0]:.4f} -> {ft_result.loss_history[-1]:.4f}")

ckpt_dir = WORK_DIR / "kambera_lora_checkpoint"
ft_result.bundle.save(ckpt_dir)
bundle = load_rsfm_checkpoint(ckpt_dir, backbone=backbone)
print(f"Checkpoint saved & reloaded from {ckpt_dir}")

## 6. Workflow 3: In-Memory Late Ancillary Fusion (`NASADEM`) -> Streaming Probabilities -> Hybrid Assembly & Olofsson Accuracy

In [ ]:
anc_list = [dem_path] if dem_path is not None else None
prob_path = infer_rsfm_probabilities(
    s2_t1_path,
    WORK_DIR / "kambera_rsfm_probs_u16.tif",
    bundle=bundle,
    chip_size=CHIP_SIZE,
    stride=STRIDE,
    block_size=256,
    output_dtype="uint16",
)

if seg_path is not None:
    hybrid_out = process_tile(
        prob_path=prob_path,
        seg_path=seg_path,
        out_dir=WORK_DIR / "hybrid_out",
        block_size=512,
        overlap=32,
    )
    lulc_map_path = hybrid_out.lulc
    print("Hybrid assembly outputs:", lulc_map_path.name, hybrid_out.stability.name, hybrid_out.ci.name)
else:
    with rasterio.open(prob_path) as src:
        probs = src.read().astype("float64") / 10000.0
        smoothed = bayesian_smooth(probs, smoothness=1.0, window_size=5)
        lulc_arr = classify_from_probs(smoothed, class_values=bundle.class_values).astype("uint8")
        lulc_map_path = WORK_DIR / "kambera_bayes_lulc.tif"
        meta = src.profile.copy()
        meta.update(count=1, dtype="uint8", nodata=0)
        with rasterio.open(lulc_map_path, "w", **meta) as dst:
            dst.write(lulc_arr, 1)

acc_result = assess_map_accuracy(
    raster_path=lulc_map_path,
    reference_data=VAL_PATH,
    class_field=CLASS_FIELD,
    class_names=KAMBERA_CLASS_NAMES,
    compute_area_adjustment=True,
)
print("Independent Validation Metrics on validation_data_kambera.shp:")
for k in ["overall_accuracy_sample", "overall_accuracy_area_weighted", "f1_macro", "kappa"]:
    if k in acc_result.overall_metrics:
        print(f"  {k}: {acc_result.overall_metrics[k]:.4f}")

## 7. Workflow 4: GEOBIA SNIC Segment Pooling (`bundle` Embeddings + `NASADEM`)

In [ ]:
if seg_path is not None:
    seg_df = pool_rsfm_by_segments(
        s2_t1_path,
        seg_path,
        bundle=bundle,
        ancillary_paths=anc_list,
        include_std=True,
        chip_size=CHIP_SIZE,
        stride=STRIDE,
        block_size=256,
    )
    labels_df = assign_segment_labels(seg_path, TRAIN_PATH, class_field=CLASS_FIELD, block_size=256)
    print(f"Pooled segments: {len(seg_df)} | Labeled segments intersecting Training_data_kambera: {len(labels_df)}")
    print(seg_df.head(4)[["segment_id", "pixel_count", "embed_000_mean"]])
else:
    print("Skipping GEOBIA segment pooling (no SNIC segment raster supplied).")

## 8. Workflow 5: Bi-Temporal Embedding Cosine Stability Mask & Sample Disturbance Filter

In [ ]:
dist_path, mask_path = compute_embedding_stability(
    s2_t1_path,
    s2_t2_path,
    WORK_DIR / "kambera_cosine_dist.tif",
    WORK_DIR / "kambera_stable_mask.tif",
    bundle=bundle,
    tau_stable=0.15,
    chip_size=CHIP_SIZE,
    stride=STRIDE,
    block_size=256,
)

clean_train, flagged_train = filter_disturbed_samples(
    TRAIN_PATH,
    dist_path,
    max_change_distance=0.15,
)
print(f"Total training points: {len(flagged_train)} | Stable: {len(clean_train)} | Disturbed: {int(flagged_train['is_disturbed'].sum())}")

## 9. Visual Inspection with Kambera Color Palette

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap

    cmap_colors = ["#000000"] + [KAMBERA_COLORS[i] for i in sorted(KAMBERA_COLORS)]
    kambera_cmap = ListedColormap(cmap_colors)

    with rasterio.open(lulc_map_path) as src_lc, \
         rasterio.open(dist_path) as src_dist, \
         rasterio.open(mask_path) as src_mask:
        fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
        axes[0].imshow(src_lc.read(1), cmap=kambera_cmap, vmin=0, vmax=len(KAMBERA_COLORS), interpolation="nearest")
        axes[0].set_title("Kambera RSFM LULC Map")
        im1 = axes[1].imshow(src_dist.read(1), cmap="inferno", vmin=0, vmax=1.0)
        axes[1].set_title("Embedding Cosine Distance (t1 -> t2)")
        fig.colorbar(im1, ax=axes[1], fraction=0.046)
        axes[2].imshow(src_mask.read(1), cmap="gray", vmin=0, vmax=1)
        axes[2].set_title("Binary Stability Mask (1=Stable)")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
except ImportError:
    print("matplotlib not installed; skipping visualization.")